# CHGIS Temporal Gazetteer (TGAZ)

This notebook demonstrates the **CHGIS Temporal Gazetteer (TGAZ)** client
in `dehergne_util.py` and shows how it can help identify Chinese place
names in the Dehergne data (Jesuits in China, 1552–1800).

**Why TGAZ?** TGAZ is the read-only REST service over the *China Historical
GIS* database (Harvard & Fudan University). Unlike Wikidata, which mostly
carries the *modern* name of a place, TGAZ records the name a place bore
**during a specific period**, together with its administrative parent and
feature type. That temporal dimension is exactly what a 17th-century
itinerary needs.

**Caveat (verified live):** the API only knows Chinese characters and
pinyin. European spellings like "Macau" or "Changchow" return nothing
useful — you must query the Chinese form (澳门) or the pinyin ("Aomen").

**Endpoint note:** the official docs still point at
`maps.cga.harvard.edu/tgaz/...`, but that host is dead (GitHub Pages
404). The service moved to `chgis.hudci.org/tgaz/...`, which the helper
uses. Docs: <https://chgis.hudci.org/tgw/#usage>

## Setup

In [1]:
from dehergne_util import (
    tgaz_query,
    tgaz_get,
    tgaz_results_to_records,
    tgaz_results_to_dataframe,
)

import pandas as pd

pd.set_option("display.max_rows", 150)
pd.set_option("display.width", 160)

## 1. Faceted search by name

The basic search takes a placename and returns every TGAZ record that
matches. Try the pinyin form of a well-known Chinese city.

In [2]:
r = tgaz_query(name="wuhu")
df = tgaz_results_to_dataframe(r)
print(r["memo"])
df[["name", "transcription", "years", "feature_type", "parent_name", "lat", "lon"]]

Results for query matching key 'wuhu%'


,name,transcription,years,feature_type,parent_name,lat,lon
0,芜湖县,Wuhu Xian,-201 ~ 221,县 (xian),鄣郡 (Zhang Jun),31.34653,118.42570
1,芜湖县,Wuhu Xian,222 ~ 372,县 (xian),丹阳郡 (Danyang Jun),31.33495,118.36262
2,芜湖县,Wuhu Xian,937 ~ 1911,县 (xian),江宁府 (Jiangning Fu),31.33495,118.36262
3,芜湖县,Wuhu Xian,1820 ~ 1820,县 (xian),太平府 (Taiping Fu),31.33495,118.36262
4,芜湖县,Wuhu Xian,1911 ~ 1911,县 (xian),太平府 (Taiping Fu),31.33495,118.36262
5,芜湖县,Wuhu Xian,1911 ~ 1911,县 (xian),太平府 (Taiping Fu),31.33495,0.00000
6,武化县,Wuhua Xian,503 ~ 593,县 (xian),,0.00000,0.00000
7,武化县,Wuhua Xian,621 ~ 1073,县 (xian),宴州 (Yan Zhou),24.07734,109.88029
8,武化县,Wuhua Xian,1086 ~ 1130,县 (xian),象州 (Xiang Zhou),24.07734,109.88029
9,五花城,Wuhuacheng,1911 ~ 1911,村镇 (cun zhen),河曲县 (Hequ Xian),39.31902,111.18810


Notice the **same place appears multiple times** under different year
ranges and administrative levels (prefecture `府` vs. county `县`, etc.).
This is the core value of a *temporal* gazetteer.

## 2. Filter by year — the key feature

Pass `year=` to keep only the records valid at a given moment. This is
what lets us ask "what was Macau called, and where did it sit
administratively, in 1660?"

In [3]:
# Macau in pinyin is "Aomen". Note: "Macau"/"Macao" return nothing useful.
r = tgaz_query(name="Aomen", year=1820)
tgaz_results_to_dataframe(r)

,name,transcription,sys_id,uri,year_from,year_to,years,parent_sys_id,parent_name,feature_type,object_type,lon,lat,data_source
0,澳门,Aomen,hvd_17143,https://chgis.hudci.org/tgaz/placename/hvd_17143,1820,1820,1820 ~ 1820,hvd_9549,广州府 (Guangzhou Fu),村镇 (cun zhen),POINT,113.552,22.2,CHGIS


In [4]:
# Guangzhou prefecture as it stood around a typical Dehergne date.
r = tgaz_query(name="Guangzhou", feature_type="fu", year=1700)
tgaz_results_to_dataframe(r)

,name,transcription,sys_id,uri,year_from,year_to,years,parent_sys_id,parent_name,feature_type,object_type,lon,lat,data_source
0,广州府,Guangzhou Fu,hvd_33047,https://chgis.hudci.org/tgaz/placename/hvd_33047,1368,1911,1368 ~ 1911,hvd_30142,江西行省 (Jiangxi Xingsh),府 (fu),POINT,113.25607,23.13462,CHGIS
1,广州府,Guangzhou Fu,hvd_97781,https://chgis.hudci.org/tgaz/placename/hvd_97781,1649,1726,1649 ~ 1726,,,府 (fu),POLYGON,0.00000,23.13462,CHGIS


## 3. Chinese characters work directly

UTF-8 Chinese characters can be sent as-is (no URL-encoding). This is
often the most reliable way to query, because several distinct pinyin
strings can map to the same characters.

A useful first step is to search a place **without** a year filter, to see
every period in which a given name was in use. Notice how the same
characters refer to different administrative units in different eras --
and how some familiar names (like Macau 澳门) only enter the record late,
because they were not Chinese administrative units during the Dehergne
period (1552-1800).

In [5]:
# Search each place across ALL years (no year filter).
for place in ["湘潭县"]:
    r = tgaz_query(name=place)
    print(f"\n{place}: {r['count of total results']} record(s)")
    display(tgaz_results_to_dataframe(r).head(5))


湘潭县: 8 record(s)


,name,transcription,sys_id,uri,year_from,year_to,years,parent_sys_id,parent_name,feature_type,object_type,lon,lat,data_source
0,湘潭县,Xiangtan Xian,hvd_41860,https://chgis.hudci.org/tgaz/placename/hvd_41860,511,588,511 ~ 588,hvd_32862,湘东郡 (Xiangdong Jun),县 (xian),POINT,113.35610,27.16591,CHGIS
1,湘潭县,Xiangtan Xian,hvd_41861,https://chgis.hudci.org/tgaz/placename/hvd_41861,589,748,589 ~ 748,hvd_32863,衡州 (Heng Zhou),县 (xian),POINT,112.95325,27.30817,CHGIS
2,湘潭县,Xiangtan Xian,hvd_41735,https://chgis.hudci.org/tgaz/placename/hvd_41735,749,1126,749 ~ 1126,hvd_32823,长沙郡 (Changsha Jun),县 (xian),POINT,112.90441,27.70402,CHGIS
3,湘潭县,Xiangtan Xian,hvd_41736,https://chgis.hudci.org/tgaz/placename/hvd_41736,1127,1294,1127 ~ 1294,hvd_32826,潭州 (Tan Zhou),县 (xian),POINT,112.91126,27.87270,CHGIS
4,湘潭县,Xiangtan Xian,hvd_41738,https://chgis.hudci.org/tgaz/placename/hvd_41738,1370,1911,1370 ~ 1911,hvd_32752,潭州府 (Tanzhou Fu),县 (xian),POINT,112.91126,27.87270,CHGIS


## 4. Other facets: feature type, parent, source

The remaining facets refine a name search rather than stand alone.
The live server returns an empty body for any query that does **not**
include a name, so `feature_type`, `year`, `source` and `parent` are
useful as filters on a name, not as browse dimensions.


In [6]:
# "Guangzhou" appears at many administrative levels and many periods.
# Narrow to prefecture (fu) units valid in 1700.
r = tgaz_query(name="Guangzhou", feature_type="fu", year=1700)
tgaz_results_to_dataframe(r)

,name,transcription,sys_id,uri,year_from,year_to,years,parent_sys_id,parent_name,feature_type,object_type,lon,lat,data_source
0,广州府,Guangzhou Fu,hvd_33047,https://chgis.hudci.org/tgaz/placename/hvd_33047,1368,1911,1368 ~ 1911,hvd_30142,江西行省 (Jiangxi Xingsh),府 (fu),POINT,113.25607,23.13462,CHGIS
1,广州府,Guangzhou Fu,hvd_97781,https://chgis.hudci.org/tgaz/placename/hvd_97781,1649,1726,1649 ~ 1726,,,府 (fu),POLYGON,0.00000,23.13462,CHGIS


## 5. Canonical record by ID

Once you have a `sys_id` (the `hvd_...` identifier) from a search, you
can fetch its full canonical record. This endpoint returns a **richer**
structure: traditional + simplified Chinese spellings, pinyin, feature
type with English translation, temporal span rules, and coordinates.

In [10]:
# hvd_33047 is Guangzhou Fu (prefecture).
record = tgaz_get("hvd_17143")
record

{'system': 'China Historical GIS, Harvard University and Fudan University',
 'license': 'CC BY-NC 4.0',
 'uri': 'https://chgis.hudci.org/tgaz/placename/hvd_17143',
 'sys_id': 'hvd_17143',
 'sys_id of alternate': '',
 'spellings': [{'written form': '澳門',
   'script': 'traditional Chinese',
   'exonym language': '',
   'attested by': '',
   'note': ''},
  {'written form': '澳门',
   'script': 'simplified Chinese',
   'exonym language': '',
   'attested by': '',
   'note': ''},
  {'written form': 'Aomen',
   'transcribed in': 'Pinyin',
   'attested by': '',
   'note': ''}],
 'feature_type': {'name': '村镇',
  'alternate name': '村鎮',
  'transcription': 'cun zhen',
  'English': 'villages and small towns'},
 'temporal': {'begin': '1820',
  'begin rule': '9',
  'end': '1820',
  'end rule': '9'},
 'spatial': {'object_type': 'POINT',
  'xy_type': 'point',
  'latitude': '22.20000',
  'longitude': '113.55200',
  'source': 'FROM_AC',
  'present_location': [{'country code': 'cn',
    'text': '澳门市内',
  

Pull out the parts you usually care about:

In [11]:
spellings = [s.get("written form") for s in record.get("spellings", [])]
ft = record.get("feature_type", {})
temp = record.get("temporal", {})
spatial = record.get("spatial", {})
print("spellings      :", spellings)
print("feature type   :", ft.get("transcription"), "|", ft.get("English"))
print("temporal span  :", temp.get("begin"), "→", temp.get("end"))
print("coordinates    :", spatial.get("latitude"), spatial.get("longitude"))

spellings      : ['澳門', '澳门', 'Aomen']
feature type   : cun zhen | villages and small towns
temporal span  : 1820 → 1820
coordinates    : 22.20000 113.55200


## 6. Mapping Dehergne place names to TGAZ

Dehergne records places in European (mostly Portuguese/Latin) spellings.
TGAZ only knows Chinese characters and pinyin, so we need a lookup from
the European spelling to the **historical** Chinese name -- which is not
always the modern one. Two findings from the raw searches above shape
this table:

* **Macau (澳门)** -- TGAZ has no record before 1820. Macau was a
  Portuguese settlement, not a Chinese administrative unit, during the
  Jesuit period, so it is genuinely absent. Use Wikidata for Macau.
* **Peking** -- the Qing capital is recorded under its prefecture name
  **顺天府 (Shuntian Fu)**, not 北京.
* **Nanking** -- recorded as **江宁府 (Jiangning Fu)** in the Qing, not 南京.

This is precisely the historical-name disambiguation that makes TGAZ
worth the extra step.

In [12]:
# European spelling -> (historical Chinese name to query in TGAZ,
#                       a Dehergne-era year,
#                       modern English label in Wikidata).
# The third column is the bridge that lets us join TGAZ (historical)
# to Wikidata (modern). Note the historical names for Peking/Nanking.
DEHERGNE_TO_TGAZ = {
    "Macau":    ("澳门",   1820, "Macau"),     # earliest TGAZ record (post-Jesuit)
    "Canton":   ("广州",   1700, "Guangzhou"),  # Guangzhou Fu
    "Hangchow": ("杭州",   1700, "Hangzhou"),  # Hangzhou Fu
    "Peking":   ("顺天府", 1700, "Beijing"),   # Shuntian Fu (the Qing capital)
    "Nanking":  ("江宁府", 1700, "Nanjing"),   # Jiangning Fu (Qing Nanjing)
}

rows = []
for euro, (chars, yr, wd_label) in DEHERGNE_TO_TGAZ.items():
    r = tgaz_query(name=chars, year=yr)
    for rec in tgaz_results_to_records(r):
        rows.append({
            "dehergne": euro,
            "tgaz_name": rec["name"],
            "transcription": rec["transcription"],
            "years": rec["years"],
            "feature_type": rec["feature_type"],
            "parent_name": rec["parent_name"],
            "tgaz_lat": rec["lat"],
            "tgaz_lon": rec["lon"],
            "wikidata_label": wd_label.lower(),  # lowercase to match Wikidata
            "sys_id": rec["sys_id"],
        })

matches = pd.DataFrame(rows)
matches

,dehergne,tgaz_name,transcription,years,feature_type,parent_name,tgaz_lat,tgaz_lon,wikidata_label,sys_id
0,Macau,澳门,Aomen,1820 ~ 1820,村镇 (cun zhen),广州府 (Guangzhou Fu),22.20000,113.55200,macau,hvd_17143
1,Canton,广州府,Guangzhou Fu,1368 ~ 1911,府 (fu),江西行省 (Jiangxi Xingsh),23.13462,113.25607,guangzhou,hvd_33047
2,Canton,广州府,Guangzhou Fu,1649 ~ 1726,府 (fu),,23.13462,0.00000,guangzhou,hvd_97781
3,Hangchow,杭州府,Hangzhou Fu,1366 ~ 1911,府 (fu),江南行省 (Jiangnan Xings),30.29412,120.16862,hangzhou,hvd_32337
4,Hangchow,杭州府,Hangzhou Fu,1366 ~ 1810,府 (fu),,30.29412,0.00000,hangzhou,hvd_90119
5,Peking,顺天府,Shuntian Fu,1403 ~ 1724,府 (fu),直隶 (Zhili),39.93143,116.36800,beijing,hvd_88188
6,Nanking,江宁府,Jiangning Fu,1645 ~ 1911,府 (fu),江南省 (Jiangnan Sheng),32.05256,118.76899,nanjing,hvd_32285
7,Nanking,江宁府,Jiangning Fu,1645 ~ 1729,府 (fu),,32.05256,0.00000,nanjing,hvd_90265


### Cross-checking TGAZ coordinates against Wikidata

We now have the same place from two independent sources:

* **TGAZ** gives the *historical* seat (e.g. the Qing prefecture
  顺天府 Shuntian Fu) with its historical coordinates.
* **Wikidata** gives the *modern* entity (e.g. Beijing, Q956) with its
  modern coordinates.

The join runs through the modern English label -- the `wikidata_label`
column added above -- matched against the Wikidata `english_label`
(case-insensitive exact match, to avoid substring false-positives like
Guangzhouwan).

We then compute the great-circle distance (`haversine_km` from
`dehergne_util`) between each TGAZ point and its Wikidata counterpart:

* **agree** -- distance under ~50 km. The historical seat and the modern
  city centre coincide, so the identification is almost certainly correct.
* **review** -- larger gap. The place may have been relocated, the
  European name may be ambiguous, or one of the coordinates is wrong.

TGAZ sometimes emits `(0, 0)` when the longitude is missing from the
source record. Those rows are flagged separately as `no TGAZ coords`
rather than counted as disagreements -- a data-quality issue, not a
geographic mismatch.

In [13]:
from dehergne_util import haversine_km, locations_wikidata_info_file

# Load the Wikidata reference file generated by location-analysis-new.ipynb.
wd = pd.read_excel(locations_wikidata_info_file)
wd = wd.rename(columns={"latitude": "wd_lat", "longitude": "wd_lon"})
# case-insensitive exact match on the English label
wd["wikidata_label"] = wd["english_label"].astype(str).str.lower()

# Attach the modern Wikidata entity to each TGAZ row.
comp = matches.merge(
    wd[["wikidata_label", "wikidata_id", "english_label",
        "wd_lat", "wd_lon"]],
    on="wikidata_label",
    how="left",
)

# Great-circle distance between TGAZ and Wikidata points.
comp["dist_km"] = comp.apply(
    lambda r: haversine_km(r.tgaz_lat, r.tgaz_lon, r.wd_lat, r.wd_lon),
    axis=1,
)


def classify(row):
    if pd.isna(row.wd_lat):
        return "no Wikidata match"
    if row.tgaz_lon == 0 or row.tgaz_lat == 0:
        return "no TGAZ coords"
    return "agree" if (row.dist_km is not None and row.dist_km < 50) else "review"


comp["agreement"] = comp.apply(classify, axis=1)

comp[[
    "dehergne", "tgaz_name", "transcription", "years",
    "english_label", "wikidata_id",
    "tgaz_lat", "tgaz_lon", "wd_lat", "wd_lon",
    "dist_km", "agreement",
]]

,dehergne,tgaz_name,transcription,years,english_label,wikidata_id,tgaz_lat,tgaz_lon,wd_lat,wd_lon,dist_km,agreement
0,Macau,澳门,Aomen,1820 ~ 1820,Macau,Q14773,22.20000,113.55200,22.190000,113.538056,1.815913,agree
1,Canton,广州府,Guangzhou Fu,1368 ~ 1911,Guangzhou,Q16572,23.13462,113.25607,23.130000,113.260000,0.652228,agree
2,Canton,广州府,Guangzhou Fu,1649 ~ 1726,Guangzhou,Q16572,23.13462,0.00000,23.130000,113.260000,11158.151970,no TGAZ coords
3,Hangchow,杭州府,Hangzhou Fu,1366 ~ 1911,Hangzhou,Q4970,30.29412,120.16862,30.250000,120.167500,4.907099,agree
4,Hangchow,杭州府,Hangzhou Fu,1366 ~ 1810,Hangzhou,Q4970,30.29412,0.00000,30.250000,120.167500,10778.413911,no TGAZ coords
5,Peking,顺天府,Shuntian Fu,1403 ~ 1724,Beijing,Q956,39.93143,116.36800,39.904030,116.407526,4.543734,agree
6,Nanking,江宁府,Jiangning Fu,1645 ~ 1911,Nanjing,Q16666,32.05256,118.76899,32.060833,118.778889,1.310179,agree
7,Nanking,江宁府,Jiangning Fu,1645 ~ 1729,Nanjing,Q16666,32.05256,0.00000,32.060833,118.778889,10416.239274,no TGAZ coords


## 7. Parameter validation

The helper validates inputs so bad queries fail fast with a clear
message.

In [14]:
# Years outside the documented range -222..1911 are rejected.
try:
    tgaz_query(name="Aomen", year=2000)
except ValueError as e:
    print("rejected ->", e)

# At least one search facet is required.
try:
    tgaz_query()
except ValueError as e:
    print("rejected ->", e)

rejected -> year 2000 outside TGAZ valid range -222..1911
rejected -> tgaz_query requires a name (the TGAZ server returns no results for queries without a placename)


## 8. Bulk match Wikidata locations to CHGIS

The previous cells matched a handful of places by hand. Here we process
**every China-related location** in the Wikidata reference file and resolve
each to its CHGIS/TGAZ record.

Two resolution paths, tried in order:

1. **Wikidata property P4711 ("CHGIS ID")** — the authoritative link.
   Fetched in batches via the `wbgetentities` API. Only ~17 of the ~290
   China locations carry it, but when present it is exact.
2. **Name + coordinate matching** — for the rest. Search TGAZ by the
   Wikidata `chinese_label` at `year=1800` (end of the Dehergne period),
   retrying with the last character stripped if the full form returns
   nothing, then pick the record nearest the Wikidata coordinates by
   haversine distance.

Non-China locations are skipped (TGAZ only covers China). The step-B
queries are rate-limited (~1.5 s each), so the full run takes several
minutes; progress is printed every 25 places.

Results are written to
`../inferences/wikidata-references/wikidata_chgis_matches.xlsx`.

In [ ]:
import time
import urllib.parse
import urllib.request

import pandas as pd

from dehergne_util import (
    haversine_km,
    locations_wikidata_info_file,
    tgaz_get,
    tgaz_query,
    tgaz_results_to_records,
)

CHGIS_MATCH_YEAR = 1800  # end of the Dehergne period (1552-1800)
OUTPUT_FILE = (
    "../inferences/wikidata-references/wikidata_chgis_matches.xlsx"
)
WD_API = "https://www.wikidata.org/w/api.php"
WD_HDRS = {
    "User-Agent": "DehergneProject/1.0 (research; jrc@macao)",
}


# --- load Wikidata reference file, keep China locations -------------------
wd_all = pd.read_excel(locations_wikidata_info_file)
wd_all["latitude"] = pd.to_numeric(wd_all["latitude"], errors="coerce")
wd_all["longitude"] = pd.to_numeric(wd_all["longitude"], errors="coerce")
china = wd_all[
    wd_all["country_label"].astype(str).str.contains(
        "China|chin|Taiwan|Qing|Ming", case=False, na=False
    )
].copy()
print(f"China locations in Wikidata file: {len(china)}")


# --- Step A: fetch P4711 (CHGIS ID) via wbgetentities, batched 50 --------
def fetch_p4711(qids, chunk_size=50, sleep_s=2):
    found = {}
    chunks = [qids[i : i + chunk_size] for i in range(0, len(qids), chunk_size)]
    for ci, chunk in enumerate(chunks, 1):
        params = {
            "action": "wbgetentities",
            "ids": "|".join(chunk),
            "props": "claims",
            "format": "json",
        }
        url = f"{WD_API}?{urllib.parse.urlencode(params)}"
        req = urllib.request.Request(url, headers=WD_HDRS)
        try:
            with urllib.request.urlopen(req, timeout=60) as resp:
                data = json.loads(resp.read().decode("utf-8"))
            for qid, ent in data.get("entities", {}).items():
                for c in ent.get("claims", {}).get("P4711", []):
                    v = c.get("mainsnak", {}).get("datavalue", {}).get("value", "")
                    if v:
                        found[qid] = str(v)
                        break
            print(f"  P4711 chunk {ci}/{len(chunks)}: cumulative {len(found)}")
        except Exception as e:
            print(f"  P4711 chunk {ci}: ERROR {e}")
        if ci < len(chunks):
            time.sleep(sleep_s)
    return found


import json  # used by fetch_p4711 above

p4711 = fetch_p4711(china["wikidata_id"].tolist())
print(f"Step A (P4711): {len(p4711)} locations linked directly")


# --- Step B: name + coordinate matching for the rest ----------------------
def _match_by_coords(zh_label, wd_lat, wd_lon, year=CHGIS_MATCH_YEAR):
    """Return (best_record, dist_km) or (None, None)."""
    if not zh_label:
        return None, None
    candidates = [zh_label]
    if len(zh_label) > 1:
        candidates.append(zh_label[:-1])  # strip 市/省/县/区/镇 suffix
    recs = []
    for label in candidates:
        try:
            r = tgaz_query(name=label, year=year)
            recs = tgaz_results_to_records(r)
        except Exception:
            recs = []
        if recs:
            break
    if not recs:
        return None, None
    scored = []
    for x in recs:
        if x["lat"] and x["lon"]:
            d = haversine_km(wd_lat, wd_lon, x["lat"], x["lon"])
            scored.append((d, x))
    if not scored:
        # records exist but none has coordinates
        return recs[0], None
    scored.sort(key=lambda t: t[0])
    return scored[0][1], scored[0][0]


def _chgis_row_from_get(rec_dict):
    """Flatten a tgaz_get canonical record into the output columns."""
    sp = rec_dict.get("spatial", {})
    ft = rec_dict.get("feature_type", {})
    tmp = rec_dict.get("temporal", {})
    return {
        "sys_id": rec_dict.get("sys_id", ""),
        "chgis_name": " ".join(
            s.get("written form", "")
            for s in rec_dict.get("spellings", [])
            if "Pinyin" in str(s.get("transcribed in", ""))
            or s.get("script") == "simplified Chinese"
        ),
        "transcription": "",
        "years": f"{tmp.get('begin','')} ~ {tmp.get('end','')}",
        "feature_type": ft.get("transcription", ""),
        "parent_name": "",
        "chgis_lat": _safe_float(sp.get("latitude")),
        "chgis_lon": _safe_float(sp.get("longitude")),
    }


def _safe_float(v):
    try:
        return float(v)
    except (TypeError, ValueError):
        return None


rows = []
todo = china[~china["wikidata_id"].isin(p4711)]
print(f"Step B (coords): {len(todo)} locations to match @ year {CHGIS_MATCH_YEAR}")
for i, (_, row) in enumerate(todo.iterrows(), 1):
    zh = str(row["chinese_label"]).strip() if pd.notna(row["chinese_label"]) else ""
    wd_lat, wd_lon = row["latitude"], row["longitude"]
    best, dist = _match_by_coords(zh, wd_lat, wd_lon)
    base = {
        "wikidata_id": row["wikidata_id"],
        "english_label": row.get("english_label", ""),
        "chinese_label": zh,
        "wd_lat": wd_lat,
        "wd_lon": wd_lon,
        "country_label": row.get("country_label", ""),
        "chgis_source": "name+coords" if best else "none",
        "sys_id": best["sys_id"] if best else "",
        "chgis_name": best["name"] if best else "",
        "transcription": best["transcription"] if best else "",
        "years": best["years"] if best else "",
        "feature_type": best["feature_type"] if best else "",
        "parent_name": best["parent_name"] if best else "",
        "chgis_lat": best["lat"] if best else None,
        "chgis_lon": best["lon"] if best else None,
        "dist_km": dist,
    }
    rows.append(base)
    if i % 25 == 0:
        print(f"  ...{i}/{len(todo)} done")
    time.sleep(1.5)  # be polite to the TGAZ server

# --- Step A rows: fetch the canonical record for each P4711 hit -----------
for qid, chgis_num in p4711.items():
    row = china[china["wikidata_id"] == qid].iloc[0]
    try:
        rec = tgaz_get(f"hvd_{chgis_num}")
        c = _chgis_row_from_get(rec)
    except Exception as e:
        c = {k: "" for k in ["sys_id","chgis_name","transcription","years","feature_type","parent_name","chgis_lat","chgis_lon"]}
        c["sys_id"] = f"hvd_{chgis_num}"
        print(f"  tgaz_get failed for {qid} hvd_{chgis_num}: {e}")
    rows.append({
        "wikidata_id": qid,
        "english_label": row.get("english_label", ""),
        "chinese_label": str(row.get("chinese_label", "") or ""),
        "wd_lat": row["latitude"],
        "wd_lon": row["longitude"],
        "country_label": row.get("country_label", ""),
        "chgis_source": "wikidata P4711",
        **c,
        "dist_km": haversine_km(row["latitude"], row["longitude"], c.get("chgis_lat"), c.get("chgis_lon")),
    })


# --- assemble, classify, write --------------------------------------------
result = pd.DataFrame(rows)


def classify(row):
    if row["chgis_source"] == "none":
        return "no TGAZ match"
    # Authoritative P4711 link: trust it; distance is only a sanity check.
    if row["chgis_source"] == "wikidata P4711":
        if pd.isna(row.get("wd_lat")):
            return "P4711 (no WD coords)"
        if row["dist_km"] is None or row["dist_km"] < 50:
            return "P4711 agree"
        return "P4711 review"
    if row.get("chgis_lat") in (None, 0) or row.get("chgis_lon") in (None, 0):
        return "no TGAZ coords"
    if row["dist_km"] is None:
        return "review"
    return "agree" if row["dist_km"] < 50 else "review"


result["agreement"] = result.apply(classify, axis=1)

cols = [
    "wikidata_id", "english_label", "chinese_label", "country_label",
    "wd_lat", "wd_lon",
    "sys_id", "chgis_name", "transcription", "years", "feature_type",
    "parent_name", "chgis_lat", "chgis_lon",
    "chgis_source", "dist_km", "agreement",
]
result = result[cols].sort_values("agreement")
result.to_excel(OUTPUT_FILE, index=False)

print(f"\nWrote {len(result)} rows to {OUTPUT_FILE}")
print(result["chgis_source"].value_counts().to_string())
print()
print(result["agreement"].value_counts().to_string())
result.head(15)

China locations in Wikidata file: 290
  P4711 chunk 1/6: cumulative 3
  P4711 chunk 2/6: cumulative 5
  P4711 chunk 3/6: cumulative 9
  P4711 chunk 4/6: cumulative 12
  P4711 chunk 5/6: cumulative 16
  P4711 chunk 6/6: cumulative 17
Step A (P4711): 17 locations linked directly
Step B (coords): 273 locations to match @ year 1800
  ...25/273 done
  ...50/273 done
  ...75/273 done
  ...100/273 done


## Summary

| Helper | Purpose |
|---|---|
| `tgaz_query(name, year, feature_type, source, parent)` | faceted search |
| `tgaz_get(sys_id)` | full canonical record by `hvd_` id |
| `tgaz_results_to_records(response)` | clean dict list from a search |
| `tgaz_results_to_dataframe(response)` | same, as a pandas DataFrame |

**When to reach for TGAZ vs Wikidata**

* **Wikidata** — best for the modern identity of a place, multilingual
  labels, and linking out to other knowledge graphs.
* **TGAZ** — best for the *historical* name and administrative position
  of a Chinese place *at a specific year*, which is exactly the question
  a 1552–1800 itinerary raises.

They are complementary: use Wikidata to anchor the modern entity, and
TGAZ to disambiguate which historical instance a Dehergne entry refers
to.